# 03 — HPO con Optuna + MLflow

En este notebook aprendemos a realizar **Hyperparameter Optimization (HPO)** usando [Optuna](https://optuna.org/), un framework moderno basado en búsqueda bayesiana, integrado con MLflow para registrar cada experimento.

**Lo que veremos:**
- Por qué el tuning de hiperparámetros importa
- Grid Search vs Random Search vs Búsqueda Bayesiana
- Optuna: Studies, Trials y samplers
- Integración Optuna + MLflow (cada trial = un run)
- Pruning para ahorrar tiempo
- Entrena el modelo final con los mejores parámetros

In [ ]:
!pip install optuna lightgbm mlflow pandas numpy scikit-learn plotly -q

In [ ]:
import os
import optuna
import mlflow
import lightgbm as lgb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
# En el contenedor de Jupyter (make up) los runs van al servidor de MLflow
# (UI: http://localhost:5050). Fuera de Docker se guardan en ./mlruns.
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "mlruns"))
print(f"Optuna: {optuna.__version__} | MLflow: {mlflow.__version__}")

## 1. El problema de los hiperparámetros

Los **hiperparámetros** son los parámetros que NO se aprenden durante el entrenamiento, sino que el desarrollador debe definir antes de entrenar el modelo. Por ejemplo:

- `learning_rate`: ¿qué tan rápido aprende el modelo?
- `n_estimators`: ¿cuántos árboles se construyen?
- `num_leaves`: ¿qué tan profundo/complejo es cada árbol?

**¿Por qué importa?** La diferencia entre hiperparámetros malos y buenos puede ser:
- Un modelo que apenas funciona vs uno que bate el estado del arte
- RMSE de 80 vs RMSE de 45 en el mismo dataset

El problema es que el espacio de hiperparámetros es enorme y continuo. No podemos probar todo manualmente.

> **Objetivo del HPO:** Encontrar la combinación de hiperparámetros que minimiza (o maximiza) la métrica de validación.

## 2. Grid vs Random vs Bayesian

Existen tres grandes enfoques para buscar los mejores hiperparámetros:

```
Grid Search:      Random Search:    Bayesian (Optuna):
┌─────────────┐   ┌─────────────┐   ┌─────────────┐
│ × × × × ×  │   │   ×   ×     │   │     ×       │
│ × × × × ×  │   │ ×       ×   │   │   ×   ×     │
│ × × × × ×  │   │     ×       │   │ ×           │
│ × × × × ×  │   │   ×   ×     │   │   ×   ××    │
│ × × × × ×  │   │ ×     ×     │   │       ×     │
└─────────────┘   └─────────────┘   └─────────────┘
Explora todo      Explora al azar   Explora más donde
(exhaustivo, lento)(más eficiente)   es prometedor
```

| Método | Ventaja | Desventaja |
|--------|---------|------------|
| Grid Search | Garantiza explorar todo | Muy lento. Maldición de la dimensionalidad |
| Random Search | Más eficiente que Grid | No aprende de los resultados anteriores |
| Bayesian (Optuna) | Aprende qué zonas son prometedoras | Algo más complejo de implementar |

**Optuna usa TPE (Tree-structured Parzen Estimator)**, un algoritmo bayesiano que modela la distribución de buenos vs malos hiperparámetros para enfocar la búsqueda.

In [ ]:
rng = np.random.default_rng(42)
n = 3000
hours = rng.integers(0, 24, n)
temps = rng.normal(26, 5, n)
demand = 1000 + 80*np.sin(2*np.pi*hours/24) + 3*temps + rng.normal(0, 50, n)

X = pd.DataFrame({"hour": hours, "temperature": temps, "is_peak": ((hours>=18)&(hours<=21)).astype(int)})
y = demand

X_train, X_val = X[:2400], X[2400:]
y_train, y_val = y[:2400], y[2400:]
print(f"Train: {len(X_train)} | Val: {len(X_val)}")

## 3. Optuna: Study y Trial — ejemplo simple

Antes de atacar LightGBM, veamos cómo funciona Optuna con una función matemática simple.

Conceptos clave:
- **Study**: el experimento de optimización completo (agrupa todos los trials)
- **Trial**: una evaluación individual con un conjunto de hiperparámetros
- **objective**: la función que queremos minimizar (o maximizar)
- `trial.suggest_*`: métodos para sugerir valores de hiperparámetros

In [ ]:
# Ejemplo con función sencilla para entender el concepto
def toy_objective(trial):
    x = trial.suggest_float("x", -10, 10)
    y = trial.suggest_float("y", -10, 10)
    return (x - 2)**2 + (y + 3)**2  # mínimo en x=2, y=-3

study = optuna.create_study(direction="minimize")
study.optimize(toy_objective, n_trials=50, show_progress_bar=False)

print(f"Mejor valor: {study.best_value:.4f}")
print(f"Mejores params: {study.best_params}")
print(f"(El mínimo real es en x=2, y=-3 \u2192 valor=0)")

## 4. HPO para LightGBM

Ahora aplicamos el mismo patrón a LightGBM. La función objetivo:
1. Recibe un `trial`
2. Sugiere valores para cada hiperparámetro
3. Entrena el modelo con esos valores
4. Evalúa en el conjunto de validación
5. Devuelve la métrica que queremos minimizar (RMSE)

In [ ]:
def lgbm_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "random_state": 42,
        "verbose": -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(20, verbose=False)])
    y_pred = model.predict(X_val)
    return float(np.sqrt(mean_squared_error(y_val, y_pred)))

study_lgbm = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study_lgbm.optimize(lgbm_objective, n_trials=15, show_progress_bar=True)

print(f"\nMejor RMSE: {study_lgbm.best_value:.2f}")
print(f"Mejores params: {study_lgbm.best_params}")

## 5. Integración con MLflow: cada trial = un run

La integración Optuna + MLflow es muy natural:
- Cada **trial** de Optuna se convierte en un **run** de MLflow
- Guardamos los hiperparámetros con `mlflow.log_params()`
- Guardamos la métrica con `mlflow.log_metric()`
- Luego podemos comparar todos los trials visualmente en la UI de MLflow

Esto nos da trazabilidad completa: sabemos exactamente qué combinación de hiperparámetros produjo cada resultado.

In [ ]:
mlflow.set_experiment("hpo_demo")

def lgbm_objective_mlflow(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 10, 80),
        "random_state": 42,
        "verbose": -1,
    }
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=False):
        mlflow.log_params(params)
        mlflow.log_param("trial_number", trial.number)

        model = lgb.LGBMRegressor(**params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(20, verbose=False)])
        y_pred = model.predict(X_val)
        rmse = float(np.sqrt(mean_squared_error(y_val, y_pred)))

        mlflow.log_metric("val_rmse", rmse)
    return rmse

study_with_mlflow = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_with_mlflow.optimize(lgbm_objective_mlflow, n_trials=10, show_progress_bar=True)

print(f"\nMejor RMSE: {study_with_mlflow.best_value:.2f}")
print(f"En MLflow puedes ver los 10 trials como runs separados!")

## 6. Visualización de la convergencia

Una vez terminado el study, podemos visualizar cómo fue mejorando el RMSE a lo largo de los trials. La línea roja muestra el **mejor RMSE acumulado** — si el algoritmo bayesiano está funcionando bien, debería mejorar más rápido que una búsqueda aleatoria.

In [ ]:
# Extraer historial de trials
trial_numbers = [t.number for t in study_with_mlflow.trials]
trial_values = [t.value for t in study_with_mlflow.trials]
best_so_far = [min(trial_values[:i+1]) for i in range(len(trial_values))]

fig = go.Figure()
fig.add_trace(go.Scatter(x=trial_numbers, y=trial_values, mode="markers",
                          name="RMSE del trial", marker=dict(color="lightblue", size=8)))
fig.add_trace(go.Scatter(x=trial_numbers, y=best_so_far, mode="lines",
                          name="Mejor RMSE hasta ahora", line=dict(color="red", width=2)))
fig.update_layout(title="Convergencia del HPO", xaxis_title="Trial #", yaxis_title="RMSE (Val)")
fig.show()

## 7. Pruning con MedianPruner

El **pruning** es una técnica para ahorrar tiempo: si un trial va por mal camino (peor que la mediana de los anteriores), se corta anticipadamente.

`MedianPruner` funciona así:
- Espera `n_startup_trials` trials para tener una mediana de referencia
- Evalúa el trial cada `n_warmup_steps` pasos
- Si el valor reportado es peor que la mediana, lanza `TrialPruned`

Esto es especialmente útil con modelos que tienen early stopping o que se pueden evaluar iterativamente.

In [ ]:
# MedianPruner: corta trials que van peor que la mediana
pruner = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=10)

pruned_study = optuna.create_study(direction="minimize", pruner=pruner,
                                     sampler=optuna.samplers.TPESampler(seed=42))

def prunable_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 10, 80),
        "random_state": 42,
        "verbose": -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    value = float(np.sqrt(mean_squared_error(y_val, y_pred)))
    trial.report(value, step=params["n_estimators"])
    if trial.should_prune():
        raise optuna.TrialPruned()
    return value

pruned_study.optimize(prunable_objective, n_trials=15, show_progress_bar=True)

n_pruned = len([t for t in pruned_study.trials if t.state == optuna.trial.TrialState.PRUNED])
n_complete = len([t for t in pruned_study.trials if t.state == optuna.trial.TrialState.COMPLETE])
print(f"\nTrials completados: {n_complete} | Podados: {n_pruned}")
print(f"El pruning ahorr\u00f3 tiempo evitando {n_pruned} trials prometedores")

## 8. Modelo final con los mejores parámetros

Una vez completado el HPO, tomamos los mejores parámetros del study y entrenamos el modelo final. Este modelo se registra en MLflow con el tag `origin: optuna_hpo` para trazabilidad.

In [ ]:
best_params = study_with_mlflow.best_params
best_params["random_state"] = 42
best_params["verbose"] = -1

mlflow.set_experiment("demand_forecasting_demo")
with mlflow.start_run(run_name="best_model_from_hpo") as run:
    mlflow.log_params(best_params)
    mlflow.set_tag("origin", "optuna_hpo")

    final_model = lgb.LGBMRegressor(**best_params)
    final_model.fit(X_train, y_train)

    y_pred_final = final_model.predict(X_val)
    rmse_final = float(np.sqrt(mean_squared_error(y_val, y_pred_final)))
    mlflow.log_metric("val_rmse", rmse_final)

    mlflow.lightgbm.log_model(final_model, "model")
    final_run_id = run.info.run_id

print(f"Modelo final entrenado con HPO. RMSE val: {rmse_final:.2f}")
print(f"Run ID: {final_run_id}")

## Resumen

En este notebook aprendimos:

- **HPO (Hyperparameter Optimization)** busca automáticamente la mejor combinación de hiperparámetros para un modelo de ML
- **Optuna** usa búsqueda bayesiana (TPE) que es más eficiente que Grid o Random Search porque aprende de los trials anteriores
- Un **Study** agrupa múltiples **Trials**, cada uno evaluando una combinación diferente de hiperparámetros
- **Integración con MLflow**: cada trial puede registrarse como un run separado, lo que permite comparar visualmente todos los experimentos
- **Pruning** con `MedianPruner` corta trials prometedores que van peor que la mediana, ahorrando tiempo de cómputo
- El flujo completo es: HPO → mejor combinación → entrenar modelo final → registrar en MLflow

**Siguiente paso:** `04_model_registry_mlflow.ipynb` — aprende a versionar y gestionar el ciclo de vida de tus modelos con el Model Registry de MLflow.